# Session 3 — What We Type in Class: pandas, continued

Last week we ended with one question to the Titanic table: `df[df["Sex"] == "female"]`. Tonight we finish the pandas moves every dataset in this course needs — counting categories, sorting, new columns, missing values, and `groupby` — first on the Titanic list, then on the Telco churn data from Session 1 and Homework 1.

**How to use it:** Runtime → Run all, then go back and edit. Every cell is one idea. If a cell surprises you, that is the cell to spend time on.

In [ ]:
# Environment check — run this first
import sys, pandas as pd
print('Python', sys.version.split()[0])
print('pandas', pd.__version__)

## Part A — Titanic, finished

### Where we left off

Load the table again and ask it the last question from Session 2.

In [ ]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/pandas-dev/pandas/master/doc/data/titanic.csv")
df.shape

In [ ]:
df.head()

In [ ]:
df[(df["Sex"] == "female") & (df["Age"] < 20)]

### Last week's "Your turn", answered

Question 3 asked for passengers under 20 in 1st class. Question 2 asked how many `Cabin` values are missing.

In [ ]:
df[(df["Age"] < 20) & (df["Pclass"] == 1)].shape[0]

In [ ]:
df["Cabin"].isnull().sum()

In [ ]:
df["Cabin"].isnull().sum() / df.shape[0]

In [ ]:
df["Cabin"].isnull().mean()   # the same fraction in one step — a mean of True/False is a share

### `describe()` — every summary number at once

Last week we called `.min()`, `.max()`, `.mean()`, `.std()` one at a time. This is all of them.

In [ ]:
df.describe()

In [ ]:
df["Age"].describe()

In [ ]:
df.describe(include="object")    # the text columns: how many unique values, and the most common one

### `value_counts()` — counting a category

The single most useful method in pandas. Ask it of every text column before you model.

In [ ]:
df["Sex"].value_counts()

In [ ]:
df["Pclass"].value_counts()

In [ ]:
df["Pclass"].value_counts(normalize=True)    # shares instead of counts

In [ ]:
df["Embarked"].value_counts()

In [ ]:
df["Embarked"].value_counts(dropna=False)    # the missing ones, counted too

In [ ]:
df["Embarked"].unique()

In [ ]:
df["Embarked"].nunique()

In [ ]:
df["Survived"].value_counts(normalize=True)   # 38% survived — the number every model has to beat

### OR, and `isin`

`&` was last week. `|` is OR. When you have a list of allowed values, `.isin()` reads better than three `|`.

In [ ]:
df[(df["Embarked"] == "C") | (df["Embarked"] == "Q")].shape[0]

In [ ]:
df[df["Embarked"].isin(["C", "Q"])].shape[0]

In [ ]:
df[~df["Embarked"].isin(["C", "Q"])].shape[0]     # ~ means NOT

### `sort_values()` — who paid the most?

Sorting a Series, sorting the whole table by a column, and sorting by two columns.

In [ ]:
df["Fare"].sort_values()

In [ ]:
df.sort_values("Fare", ascending=False).head(10)

In [ ]:
df.sort_values(["Pclass", "Fare"], ascending=[True, False]).head(10)

In [ ]:
df.sort_values("Fare", ascending=False)[["Name", "Pclass", "Fare"]].head(5)

### `loc` and `iloc` — rows and columns by name, or by position

`loc[rows, columns]` uses labels. `iloc[rows, columns]` uses integer positions. The comma separates rows from columns.

In [ ]:
df.loc[0]

In [ ]:
df.loc[0:4, ["Name", "Age", "Fare"]]

In [ ]:
df.loc[df["Age"] < 20, ["Name", "Age"]]

In [ ]:
df.iloc[0:5, 0:4]

In [ ]:
df.iloc[-1]    # the last row

### New columns — a feature is a hypothesis

Write to a new column name and it appears. `SibSp + Parch + 1` is the size of the family travelling together. `Age < 18` is True/False, which pandas treats as 1/0 when you take a mean.

In [ ]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df[["Name", "SibSp", "Parch", "FamilySize"]].head()

In [ ]:
df["IsChild"] = df["Age"] < 18
df["IsChild"].value_counts()

In [ ]:
df["IsChild"].mean()

In [ ]:
df["FamilySize"].value_counts().sort_index()

### Missing values — three choices

`df.isnull().sum()` tells you where. Then, per column, you choose: drop the rows, fill the value, or keep the missingness as a feature.

- `Age`: 177 missing out of 891. Fill with the median — the mean is pulled up by a few old passengers.
- `Cabin`: 687 missing. Filling is meaningless. But *having* a cabin number is information: keep it as a flag.
- `Embarked`: 2 missing. Drop the rows or fill with the most common port — it will not matter.

In [ ]:
df.isnull().sum()

In [ ]:
df["Age"].mean(), df["Age"].median()

In [ ]:
df["Age"].hist(bins=30);

In [ ]:
df.dropna().shape      # drop any row with any missing value — look what it costs

In [ ]:
df["Age_filled"] = df["Age"].fillna(df["Age"].median())
df[["Age", "Age_filled"]].isnull().sum()

In [ ]:
df["HasCabin"] = df["Cabin"].notnull()
df["HasCabin"].value_counts()

In [ ]:
df["Embarked"].value_counts().index[0]

In [ ]:
df["Embarked"] = df["Embarked"].fillna("S")
df["Embarked"].isnull().sum()

### `groupby` — one number per group

Last week's "largest so far" loop computed a maximum by walking the list. `groupby` walks the table for you: split the rows into groups, compute something in each, glue the answers back together.

`df.groupby("Sex")["Survived"].mean()` reads as: for each value of Sex, the mean of Survived. Because Survived is 0/1, the mean is the survival rate.

In [ ]:
df.groupby("Sex")["Survived"].mean()

In [ ]:
df.groupby("Pclass")["Survived"].mean()

In [ ]:
df.groupby("Pclass")["Fare"].mean()

In [ ]:
df.groupby("Pclass")["Age"].median()

In [ ]:
df.groupby("Pclass").size()

In [ ]:
df.groupby("Pclass")["Survived"].agg(["mean", "count"])

In [ ]:
df.groupby(["Sex", "Pclass"])["Survived"].mean()

In [ ]:
df.groupby(["Sex", "Pclass"])["Survived"].mean().unstack()

In [ ]:
df.groupby("HasCabin")["Survived"].mean()     # the flag we built from missingness carries signal

In [ ]:
df.groupby("FamilySize")["Survived"].agg(["mean", "count"])

In [ ]:
pd.crosstab(df["Pclass"], df["Survived"])

In [ ]:
df.groupby("Pclass")["Survived"].mean().plot(kind="bar");

### Your turn — Titanic (10 minutes)

1. What is the average fare paid by passengers who survived, compared with those who did not? One `groupby` line.
2. Which embarkation port had the highest survival rate? Show the table and name the port.
3. Build a column `Title` from `Name` with `df["Name"].str.split(",").str[1].str.split(".").str[0].str.strip()`, then `value_counts()` it. What are the four most common titles?
4. Survival rate by `Title`, for titles with at least 10 passengers. Which title survived least?

In [ ]:
# Your turn — work here

## Part B — the same moves on Telco

Titanic was practice. This is the course dataset: 7,043 customers of a telecom company, and whether each one left. Everything from Part A applies unchanged — and this is the table Homework 1 asked you about.

In [ ]:
url = "https://raw.githubusercontent.com/harslan/isom-835/master/public/data/telco_churn.csv"
telco = pd.read_csv(url)
telco.shape

In [ ]:
telco.head()

In [ ]:
telco.info()

In [ ]:
telco["Churn"].value_counts()

In [ ]:
telco["Churn"].value_counts(normalize=True)    # 26.5% churn — the baseline

### The `TotalCharges` gotcha

`.info()` says `TotalCharges` is text (`object`), not a number. `pd.to_numeric(..., errors="coerce")` converts what it can and turns the rest into missing — then `isnull()` shows you exactly which rows could not convert.

In [ ]:
telco["TotalCharges"].head()

In [ ]:
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce")
telco["TotalCharges"].isnull().sum()

In [ ]:
telco.loc[telco["TotalCharges"].isnull(), ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]

In [ ]:
# All eleven have tenure 0: brand-new customers who have not been billed yet. Zero is the honest fill.
telco["TotalCharges"] = telco["TotalCharges"].fillna(0)
telco["TotalCharges"].isnull().sum()

### A 0/1 target, so `groupby` means "churn rate"

`Churn` is "Yes"/"No". Comparing to "Yes" gives True/False; `.astype(int)` makes it 1/0. Now the mean of any group is that group's churn rate.

In [ ]:
telco["churned"] = (telco["Churn"] == "Yes").astype(int)
telco["churned"].mean()

In [ ]:
telco.groupby("Contract")["churned"].mean()

In [ ]:
telco.groupby("Contract")["churned"].agg(["mean", "count"]).sort_values("mean", ascending=False)

In [ ]:
telco.groupby("InternetService")["churned"].mean().sort_values(ascending=False)

In [ ]:
telco.groupby("PaymentMethod")["churned"].mean().sort_values(ascending=False)

In [ ]:
telco.groupby(["Contract", "PaymentMethod"])["churned"].mean().unstack()

In [ ]:
telco.groupby(["Contract", "PaymentMethod"])["churned"].agg(["mean", "count"]).sort_values("mean", ascending=False).head(5)

### Numbers into buckets: `pd.cut`

Tenure runs 0–72 months. Churn by *month* is 73 tiny groups; churn by *bucket* is a story.

In [ ]:
telco["tenure"].describe()

In [ ]:
telco["tenure_bucket"] = pd.cut(telco["tenure"], bins=[-1, 6, 12, 24, 48, 72],
                                labels=["0-6", "7-12", "13-24", "25-48", "49-72"])
telco["tenure_bucket"].value_counts().sort_index()

In [ ]:
telco.groupby("tenure_bucket", observed=True)["churned"].mean()

In [ ]:
telco.groupby("tenure_bucket", observed=True)["churned"].mean().plot(kind="bar");

### Engineering two features — the Homework 1 questions

Each new column is a guess about *why* customers leave.

- `charges_per_month`: total charges divided by tenure. For tenure 0 that would divide by zero, so use `MonthlyCharges` there.
- `n_addons`: how many of the six add-on services the customer has. Count the "Yes" values across those columns.

In [ ]:
telco["charges_per_month"] = telco["TotalCharges"] / telco["tenure"].replace(0, 1)
telco[["tenure", "MonthlyCharges", "TotalCharges", "charges_per_month"]].head()

In [ ]:
addons = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
telco[addons].head()

In [ ]:
(telco[addons] == "Yes").head()

In [ ]:
telco["n_addons"] = (telco[addons] == "Yes").sum(axis=1)    # axis=1: sum across the columns, one number per row
telco["n_addons"].value_counts().sort_index()

In [ ]:
telco.groupby("n_addons")["churned"].agg(["mean", "count"])

In [ ]:
telco.groupby("churned")[["tenure", "MonthlyCharges", "charges_per_month", "n_addons"]].mean()

### The decision: who do you call first?

A retention team can call 500 customers this week. Sort the table by the segment churn rate and the answer is the top of the list — month-to-month, electronic check, first six months.

In [ ]:
segment = telco.groupby(["Contract", "PaymentMethod", "tenure_bucket"], observed=True)["churned"].agg(["mean", "count"])
segment.sort_values("mean", ascending=False).head(10)

In [ ]:
call_first = telco[(telco["Contract"] == "Month-to-month") &
                   (telco["PaymentMethod"] == "Electronic check") &
                   (telco["tenure"] <= 6)]
call_first.shape[0], call_first["churned"].mean()

In [ ]:
call_first[["customerID", "tenure", "MonthlyCharges", "n_addons", "Churn"]].sort_values("MonthlyCharges", ascending=False).head(10)

### Your turn — Telco (15 minutes)

1. Churn rate by `SeniorCitizen`. Then by `SeniorCitizen` *and* `Contract`, unstacked. Where is the worst cell?
2. Build `has_partner_or_dependents` (True if `Partner` is "Yes" or `Dependents` is "Yes"). Churn rate with and without.
3. Among customers with fiber-optic internet, does `TechSupport` change the churn rate? One `groupby` on a filtered table.
4. Which single `PaymentMethod` × `InternetService` combination has the most churned customers in *count*, not rate? Why might the business care about count and rate differently?

In [ ]:
# Your turn — work here

## What we did tonight

- `describe()`, `value_counts()`, `unique()` — know the column before you use it.
- `|`, `~`, `isin()` — filters that read like the question you meant.
- `sort_values()`, `loc`, `iloc` — find the rows you want, by label or by position.
- New columns are hypotheses. Missing values are a choice: drop, fill, or flag.
- `groupby(...)[col].mean()` on a 0/1 column is a rate per group. `agg(["mean", "count"])` keeps you honest about small groups.
- `pd.cut` turns a number into a story. `pd.crosstab` counts two categories at once.

**Next:** every `fillna`, every `pd.cut`, every new column we typed tonight learns something from the data. In scikit-learn those steps become a `Pipeline`, so they are fit on the training rows only. That is the Session 3 notebook on the session page — open it when you are ready for the model.